In [34]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

file_path = "../data/raw/logical_fallacy_dataset.xlsx"
excel = pd.ExcelFile(file_path)

df = pd.read_excel(file_path, sheet_name="Dataset")

# Separate features and labels
X_train = df[df["split"] == "train"]["argument"]
X_val = df[df["split"] == "val"]["argument"]
X_test = df[df["split"] == "test"]["argument"]

label_columns = [
    col for col in df.columns
    if col not in [
        "id", "argument", "fallacies",
        "num_fallacies", "split",
        "char_count", "word_count", "cleaned_words"
    ]
]

y_train = df[df["split"] == "train"][label_columns]
y_val = df[df["split"] == "val"][label_columns]
y_test = df[df["split"] == "test"][label_columns]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (14341,) (14341, 16)
Validation: (1768,) (1768, 16)
Testing: (1757,) (1757, 16)


In [35]:
%pip install tokenizers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase

In [37]:
# Initialise BPE
bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# Convert text to lowercase
bpe_tokenizer.normalizer = Lowercase()

# Separate words and punctuation
bpe_tokenizer.pre_tokenizer = Whitespace()

# Configure the BPE trainer
bpe_trainer = BpeTrainer(
    vocab_size=5000,
    min_frequency=2,
    special_tokens=["[UNK]"]
)

# Train using training arguments only
bpe_tokenizer.train_from_iterator(
    X_train.tolist(),
    trainer=bpe_trainer
)

print("BPE vocabulary size:", bpe_tokenizer.get_vocab_size())

BPE vocabulary size: 5000


In [38]:
sample = X_train.iloc[0]

print("Original argument:")
print(sample)

print("\nBPE tokens:")
print(bpe_tokenizer.encode(sample).tokens)

Original argument:
A politician declares: 'Recently, there have been two criminal cases in our town involving outsiders. Clearly, all immigrants are dangerous! To protect our safety, we must immediately ban all outsiders from entering, or else your families will be in danger!'

BPE tokens:
['a', 'politician', 'decl', 'are', 's', ':', "'", 'recently', ',', 'there', 'have', 'been', 'two', 'criminal', 'cases', 'in', 'our', 'town', 'invol', 'ving', 'out', 'si', 'ders', '.', 'clearly', ',', 'all', 'immigr', 'ants', 'are', 'dangerous', '!', 'to', 'protect', 'our', 'safety', ',', 'we', 'must', 'immediately', 'ban', 'all', 'out', 'si', 'ders', 'from', 'ent', 'ering', ',', 'or', 'else', 'your', 'families', 'will', 'be', 'in', 'danger', "!'"]


In [39]:
words = [
    "calculators",
    "calculation",
    "mathematical",
    "unreasonable",
    "misinformation"
]

for word in words:
    print(word, "->", bpe_tokenizer.encode(word).tokens)

calculators -> ['calculators']
calculation -> ['calculation']
mathematical -> ['mathematical']
unreasonable -> ['unreasonable']
misinformation -> ['mis', 'information']


In [40]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase

configs = [
    (2000, 2),
    (5000, 2),
    (8000, 2),
    (5000, 5),
    (5000, 10)
]

results = []

for vocab_size, min_frequency in configs:

    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.normalizer = Lowercase()
    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        special_tokens=["[UNK]"]
    )

    tokenizer.train_from_iterator(
        X_train.tolist(),
        trainer=trainer
    )

    # Tokenise all training arguments
    encodings = tokenizer.encode_batch(X_train.tolist())

    total_tokens = sum(len(enc.tokens) for enc in encodings)
    average_tokens = total_tokens / len(encodings)

    results.append({
        "vocab_size_requested": vocab_size,
        "min_frequency": min_frequency,
        "actual_vocab_size": tokenizer.get_vocab_size(),
        "average_tokens_per_argument": average_tokens
    })

import pandas as pd

results_df = pd.DataFrame(results)
print(results_df)

   vocab_size_requested  min_frequency  actual_vocab_size  \
0                  2000              2               2000   
1                  5000              2               5000   
2                  8000              2               8000   
3                  5000              5               5000   
4                  5000             10               4730   

   average_tokens_per_argument  
0                    38.433791  
1                    33.624433  
2                    32.668852  
3                    33.624433  
4                    33.788369  


In [41]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase

bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

bpe_tokenizer.normalizer = Lowercase()
bpe_tokenizer.pre_tokenizer = Whitespace()

bpe_trainer = BpeTrainer(
    vocab_size=5000,
    min_frequency=2,
    special_tokens=["[UNK]"]
)

bpe_tokenizer.train_from_iterator(
    X_train.tolist(),
    trainer=bpe_trainer
)

In [42]:
X_train_bpe = [
    " ".join(encoding.tokens)
    for encoding in bpe_tokenizer.encode_batch(X_train.tolist())
]

X_val_bpe = [
    " ".join(encoding.tokens)
    for encoding in bpe_tokenizer.encode_batch(X_val.tolist())
]

X_test_bpe = [
    " ".join(encoding.tokens)
    for encoding in bpe_tokenizer.encode_batch(X_test.tolist())
]

In [43]:
from sklearn.feature_extraction.text import CountVectorizer

bpe_vectorizer = CountVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False
)

X_train_bow_bpe = bpe_vectorizer.fit_transform(X_train_bpe)

X_val_bow_bpe = bpe_vectorizer.transform(X_val_bpe)

X_test_bow_bpe = bpe_vectorizer.transform(X_test_bpe)

print("Training:", X_train_bow_bpe.shape)
print("Validation:", X_val_bow_bpe.shape)
print("Testing:", X_test_bow_bpe.shape)

Training: (14341, 4615)
Validation: (1768, 4615)
Testing: (1757, 4615)


In [44]:
from sklearn.feature_extraction.text import CountVectorizer

word_vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95
)

# Learn vocabulary from training data
X_train_bow_word = word_vectorizer.fit_transform(X_train)

# Apply the same vocabulary to validation and test
X_val_bow_word = word_vectorizer.transform(X_val)
X_test_bow_word = word_vectorizer.transform(X_test)

print("Word-level BoW")
print("Training:", X_train_bow_word.shape)
print("Validation:", X_val_bow_word.shape)
print("Testing:", X_test_bow_word.shape)

print("\nVocabulary size:", len(word_vectorizer.vocabulary_))

Word-level BoW
Training: (14341, 6035)
Validation: (1768, 6035)
Testing: (1757, 6035)

Vocabulary size: 6035


In [45]:
comparison = pd.DataFrame({
    "Representation": ["Word-level BoW", "BPE Subword BoW"],
    "Training samples": [
        X_train_bow_word.shape[0],
        X_train_bow_bpe.shape[0]
    ],
    "Features": [
        X_train_bow_word.shape[1],
        X_train_bow_bpe.shape[1]
    ]
})

print(comparison)

    Representation  Training samples  Features
0   Word-level BoW             14341      6035
1  BPE Subword BoW             14341      4615


In [46]:
print("Word-level vocabulary:")
print(list(word_vectorizer.vocabulary_.keys())[:30])

print("\nBPE vocabulary:")
print(bpe_vectorizer.get_feature_names_out()[:30])

Word-level vocabulary:
['politician', 'declares', 'recently', 'there', 'have', 'been', 'two', 'criminal', 'cases', 'in', 'our', 'town', 'involving', 'clearly', 'all', 'immigrants', 'are', 'dangerous', 'to', 'protect', 'safety', 'we', 'must', 'immediately', 'ban', 'from', 'entering', 'or', 'else', 'your']

BPE vocabulary:
['!' '!"' "!'" '"' '$' '%' '%,' '%.' '&' "'" "'," '(' ')' '),' '+' ','
 ",'" '-' '.' '."' ".'" '/' '0' '00' '000' '1' '10' '100' '15' '2']


In [47]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

word_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight=None
    )
)

word_model.fit(X_train_bow_word, y_train)

print("Word-level model trained successfully!")

Word-level model trained successfully!


In [48]:
bpe_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight=None
    )
)

bpe_model.fit(X_train_bow_bpe, y_train)

print("BPE model trained successfully!")

BPE model trained successfully!


In [49]:
# Word-level predictions
y_pred_word = word_model.predict(X_val_bow_word)

# BPE subword predictions
y_pred_bpe = bpe_model.predict(X_val_bow_bpe)

print("Word-level predictions:", y_pred_word.shape)
print("BPE predictions:", y_pred_bpe.shape)

Word-level predictions: (1768, 16)
BPE predictions: (1768, 16)


In [50]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss
)

def evaluate_model(y_true, y_pred, model_name):

    print(f"\n{model_name}")
    print("-" * 40)

    print("Micro F1:", f1_score(y_true, y_pred, average="micro", zero_division=0))
    print("Macro F1:", f1_score(y_true, y_pred, average="macro", zero_division=0))
    print("Weighted F1:", f1_score(y_true, y_pred, average="weighted", zero_division=0))

    print("Micro Precision:", precision_score(y_true, y_pred, average="micro", zero_division=0))
    print("Micro Recall:", recall_score(y_true, y_pred, average="micro", zero_division=0))

    print("Hamming Loss:", hamming_loss(y_true, y_pred))


evaluate_model(y_val, y_pred_word, "Word-level BoW + Logistic Regression")

evaluate_model(y_val, y_pred_bpe, "BPE Subword BoW + Logistic Regression")


Word-level BoW + Logistic Regression
----------------------------------------
Micro F1: 0.7974068071312804
Macro F1: 0.7641158168265758
Weighted F1: 0.7947637199022471
Micro Precision: 0.8614307153576788
Micro Recall: 0.7422413793103448
Hamming Loss: 0.030931843891402716

BPE Subword BoW + Logistic Regression
----------------------------------------
Micro F1: 0.7985364738166019
Macro F1: 0.7714877086297096
Weighted F1: 0.7968948505297462
Micro Precision: 0.8504627374573794
Micro Recall: 0.7525862068965518
Hamming Loss: 0.031143947963800905


In [51]:
from sklearn.metrics import classification_report

print("WORD-LEVEL BoW")
print(classification_report(
    y_val,
    y_pred_word,
    target_names=label_columns,
    zero_division=0
))

print("\nBPE SUBWORD BoW")
print(classification_report(
    y_val,
    y_pred_bpe,
    target_names=label_columns,
    zero_division=0
))

WORD-LEVEL BoW
                          precision    recall  f1-score   support

          Slippery Slope       0.89      0.88      0.89       490
     Appeal to Authority       0.87      0.80      0.83       279
  Bandwagon / Popularity       0.75      0.70      0.72       262
    Hasty Generalization       0.80      0.65      0.72       213
 False Cause / Causality       0.89      0.76      0.82       211
       Appeal to Emotion       0.88      0.71      0.78       207
             Composition       0.93      0.77      0.84       151
           False Dilemma       0.87      0.66      0.75       139
              Ad Hominem       0.83      0.56      0.67       129
            Equivocation       0.95      0.69      0.80        81
      Circular Reasoning       0.87      0.74      0.80        73
               Straw Man       0.86      0.62      0.72        39
             Red Herring       1.00      0.53      0.69        19
     Appeal to Ignorance       0.83      0.50      0.62     